# Title
Model Comparison Report
## Purpose
Compare benchmark runs by loading saved metrics and predictions instead of retraining anything.
## Inputs
`results/benchmarks/**/metrics.json` and the matching `predictions.csv` files when they exist.
## Outputs
A compact comparison table and a best-model summary for reviewers.
## Key Takeaways
This notebook is read-only by default and only reports what the benchmark runs already produced.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RESULTS_DIR

BENCHMARKS_DIR = RESULTS_DIR / 'benchmarks'

def newest_benchmark_run_dir(root: Path) -> Path | None:
    run_dirs = []
    for metrics_file in root.glob('**/metrics.json'):
        run_dir = metrics_file.parent
        if (run_dir / 'predictions.csv').exists():
            run_dirs.append(run_dir)
    if not run_dirs:
        return None
    return max(run_dirs, key=lambda path: path.stat().st_mtime)


In [ ]:
BENCHMARK_RUN_DIR = newest_benchmark_run_dir(BENCHMARKS_DIR)
metrics_path = BENCHMARK_RUN_DIR / 'metrics.json' if BENCHMARK_RUN_DIR is not None else None
predictions_path = BENCHMARK_RUN_DIR / 'predictions.csv' if BENCHMARK_RUN_DIR is not None else None

print(f'Benchmarks directory: {BENCHMARKS_DIR}')
print(f'Benchmark run directory: {BENCHMARK_RUN_DIR}')
print(f'Latest metrics file: {metrics_path}')
print(f'Latest predictions file: {predictions_path}')

metrics_df = pd.DataFrame()
predictions_df = pd.DataFrame()

if metrics_path is not None:
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    metrics_df = pd.DataFrame(metrics).T.reset_index().rename(columns={'index': 'model_name'})
    print('\nLoaded metrics summary:')
    print(metrics_df.sort_values('mae_eur').to_string(index=False))

if predictions_path is not None:
    predictions_df = pd.read_csv(predictions_path)
    print('\nPrediction sample:')
    print(predictions_df.head().to_string(index=False))


In [ ]:
if not metrics_df.empty:
    best_row = metrics_df.sort_values('mae_eur').iloc[0]
    print(
        f"Best model by MAE: {best_row['model_name']} (mae_eur={best_row['mae_eur']:.2f}, within_15pct={best_row['within_15pct']:.3f})"
    )
else:
    print('No metrics were found, so there is nothing to compare yet.')

if not predictions_df.empty and 'model_name' in predictions_df.columns:
    residual_summary = predictions_df.groupby('model_name', as_index=False)['residual_eur'].agg(['mean', 'median']).reset_index()
    print('\nResidual summary by model:')
    print(residual_summary.to_string(index=False))
